# Photo-z: confronto tra modelli per una stima accurata 

## Importazione delle librerie

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
import tensorflow as tf
import tensorflow_probability as tfp
import gpflow as gpf
from tqdm import tqdm

## Caricamento dataset e rimozione colonna index

In [ ]:
df_train_noCS = pd.read_csv("data/train_noCS.csv")
if 'index' in df_train_noCS.columns:
    df_train_noCS = df_train_noCS.drop(['index'], axis=1)
    
df_test_noCS = pd.read_csv("data/test_noCS.csv")
if 'index' in df_test_noCS.columns:
    df_test_noCS = df_test_noCS.drop(['index'], axis=1)

df_train_weakCS = pd.read_csv("data/train_weakCS.csv")
if 'index' in df_train_weakCS.columns:
    df_train_weakCS = df_train_weakCS.drop(['index'], axis=1)

df_test_weakCS = pd.read_csv("data/test_weakCS.csv")
if 'index' in df_test_weakCS.columns:
    df_test_weakCS = df_test_weakCS.drop(['index'], axis=1)

df_train_mildCS = pd.read_csv("data/train_mildCS.csv")
if 'index' in df_train_mildCS.columns:
    df_train_mildCS = df_train_mildCS.drop(['index'], axis=1)

df_test_mildCS = pd.read_csv("data/test_mildCS.csv")
if 'index' in df_test_mildCS.columns:
    df_test_mildCS = df_test_mildCS.drop(['index'], axis=1)

df_train_strongCS = pd.read_csv("data/train_strongCS.csv")
if 'index' in df_train_strongCS.columns:
    df_train_strongCS = df_train_strongCS.drop(['index'], axis=1)

df_test_strongCS = pd.read_csv("data/test_strongCS.csv")
if 'index' in df_test_strongCS.columns:
    df_test_strongCS = df_test_strongCS.drop(['index'], axis=1)

In [ ]:
dfs_train = {"df_train_noCS":df_train_noCS,
             "df_train_weakCS":df_train_weakCS,
             "df_train_mildCS":df_train_mildCS,
             "df_train_strongCS":df_train_strongCS}

dfs_test = {"df_test_noCS":df_test_noCS,
            "df_test_weakCS":df_test_weakCS,
            "df_test_mildCS":df_test_mildCS,
            "df_test_strongCS":df_test_strongCS}


## Exploratory Data Analysis

Intanto capiamo il significato visuale della CS nei datasets. Si può notare come più è elevata la CS, e più le distribuzioni di train e test delle varie features siano diverse. Vediamolo graficamente:

In [ ]:
#Definisco le features da plottare (sapendo che sono uguali per tutti i datasets)
features = df_train_noCS.columns.tolist()

# Organizzo i dataset per livello di CS
dataset_groups = {
    'noCS': {'train': df_train_noCS, 'test': df_test_noCS},
    'weakCS': {'train': df_train_weakCS, 'test': df_test_weakCS},
    'mildCS': {'train': df_train_mildCS, 'test': df_test_mildCS},
    'strongCS': {'train': df_train_strongCS, 'test': df_test_strongCS}
}

cs_levels = ['noCS', 'weakCS', 'mildCS', 'strongCS']

#Ciclo su ogni feature e creo una figura dedicata con dei subplots
for feature in features:
    fig, axes = plt.subplots(nrows=len(cs_levels), ncols=2, figsize=(12, 4 * len(cs_levels)), sharex=True)
    fig.suptitle(f'Distribuzione della Feature: {feature} attraverso i livelli di CS (Train vs Test)', fontsize=16, y=1.02)

    for i, cs_level in enumerate(cs_levels):
        train_df = dataset_groups[cs_level]['train']
        test_df = dataset_groups[cs_level]['test']

        #Plot distribuzione di training
        sns.histplot(train_df[feature], kde=True, ax=axes[i, 0], color='skyblue', label='Train', stat='density')
        axes[i, 0].set_title(f'{cs_level} Train', fontsize=10)
        axes[i, 0].set_ylabel('Density')
        axes[i, 0].legend()

        #Plot distribuzione di test
        sns.histplot(test_df[feature], kde=True, ax=axes[i, 1], color='salmon', label='Test', stat='density')
        axes[i, 1].set_title(f'{cs_level} Test', fontsize=10)
        axes[i, 1].set_ylabel('Density')
        axes[i, 1].legend()
        axes[i, 1].set_xlabel('') 

    #Imposto una label solo per i plots di riga più bassa.
    for j in range(2):
        axes[len(cs_levels)-1, j].set_xlabel(feature)

    plt.tight_layout(rect=[0, 0.03, 1, 0.98]) 
    plt.show()

In effetti, si osserva come le distribuzioni delle features si differenzino sempre di più, via via che aumenta il grado di CS nel dataset. Questo ammonisce sul fatto che le prestazioni dei modelli che utilizzeremo dovrebbero tendere a peggiorare nel momento in cui aumentiamo il grado di CS presente.

L'obiettivo sarà dunque quello di individuare il modello (tra quelli che prenderemo in considerazione) che mantenga le sue prestazioni più stabili possibili all'aumentare di CS.  

## Visualizzo una descrizione riassuntiva dei dataset

In [ ]:
for name, df in dfs_train.items():
    print(f"Dataset: {name}")
    print(df.describe())
    print("\n")

Si nota come tutte le feature siano dello stesso ordine di grandezza, tranne la prima. Andrò allora a standardizzarla, non tanto perché sia indispensabile per i modelli che andrò a testare, ma per facilitare il confronto tra l'importanza delle features (che ad esempio per la regressione lineare, si basa sui coefficienti delle stesse). 

## Definisco alcune funzioni utili

In [ ]:
def extract_yX(df):
    y = df['Z']
    X = df.drop(['Z'], axis=1)
    return y , X 

def standardize_data(X_tr, X_te):
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_te = scaler.transform(X_te)
    return X_tr, X_te

def calculate_errors(y_te, y_pred):
    mse = mean_squared_error(y_te, y_pred)
    rmse = root_mean_squared_error(y_te, y_pred)
    return mse, rmse
    

## Plot del valori di training di ciascuna feature in relazione al valore del target. Cerco pattern evidenti visualmente.

Analizziamo graficamente la relazione tra features e target, una feature alla volta.

In [ ]:
for name, df in dfs_train.items():
    print(f"Dataset: {name}")
    
    #Ottengo la lista delle features 
    y_tr, X_tr = extract_yX(df)
    feature_names = X_tr.columns

    #Creo figure per contenere tutti i plot
    plt.figure(figsize=(18, 12)) # Aggiusto la dimensione della figure size 

    #Loop attraverso ogni feature e creo uno scatter plot rispetto y_tr
    for i, feature in enumerate(feature_names):
        plt.subplot(2, 3, i + 1) # Predispongo i plots in una griglia 2x3
        sns.scatterplot(x=X_tr[feature], y=y_tr, alpha=0.5)
        plt.title(f'Feature: {feature} vs Target: Z')
        plt.xlabel(feature)
        plt.ylabel('Z')
        plt.grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout() 
    plt.show()

Si osserva come le relazioni siano tutte evedentemente non lineari, cosa che rende ideale l'utilizzo di modelli complessi. 
Utilizzeremo ad ogni modo anche la regressione lineare come baseline, anche se ci aspettiamo performance non elevate.
Si nota dai grafici anche come le forme non siano nette e pulite, ma abbastanza rumorose. Questo significa che una spiegazione univariata è utile, ma che ci sono evidenti interazioni tra le variabili.

Appare anche abbastanza evidente come la dispersione di Z cambi al variare dei valori della feature, cosa che è indice di una possibile varianza che non è costante, ma che invece vari. Questo è un segnale di eteroschedasticità, anche se su questo ci torneremo più avanti per una valutazione più accurata.

Infine, è lampante come alcune features abbiano struttura abbastanza chiara, mentre altre siano più rumorose. Questo ci fa capire come alcune siano più informative, ed altre lo siano meno.


# Dataframes in cui conservare i risultati (MSE e RMSE)

In [ ]:
results_mse = pd.DataFrame(columns=['Modello','test_noCS','test_weakCS','test_mildCS','test_strongCS'])
results_rmse = pd.DataFrame(columns=['Modello','test_noCS','test_weakCS','test_mildCS','test_strongCS'])

overfitting_results = pd.DataFrame(columns=['Modello', 'train_noCS', 'val_noCS', 'train_weakCS', 'val_weakCS',
                                            'train_mildCS', 'val_mildCS', 'train_strongCS', 'val_strongCS'])

# Baseline banale per confrontare i modelli

In [ ]:
row_mse_results = ["Baseline"]
row_rmse_results = ["Baseline"]
for name, group in dataset_groups.items():
    y_tr, X_tr = extract_yX(group["train"])
    y_te, X_te = extract_yX(group['test'])
    
    y_mean = y_tr.mean()
    y_pred_baseline = y_mean * np.ones(y_te.shape)


    mse, rmse = calculate_errors(y_te, y_pred_baseline)
   
    row_mse_results.append(mse)
    row_rmse_results.append(rmse)

results_mse.loc[len(results_mse)] = row_mse_results
results_rmse.loc[len(results_rmse)] = row_rmse_results

display(results_mse)
display(results_rmse)

# Regressione Lineare

Definisco un modello di regressione lineare, e vado poi a fare cross validation per valutare la presenza di overfitting o meno.

In [ ]:
row_mse_results = ["Regressione Lineare"]
row_rmse_results = ["Regressione Lineare"]
row_overfitting_results = ["Regressione Lineare"]

#Serve poi per explainability
coefficients_importance = []

for name, group in dataset_groups.items():
    y_tr, X_tr = extract_yX(group["train"])
    y_te, X_te = extract_yX(group['test'])

    X_tr, X_te = standardize_data(X_tr, X_te)
    
    kf = KFold(n_splits=5, shuffle=True)
    reg = LinearRegression()
    scores = cross_validate(reg, X_tr, y_tr, cv=kf, scoring="neg_mean_squared_error", return_train_score=True)

    #Studio dell'overfitting usando cross validation. DECIDERE DOVE METTERLO PER EVITARE QUESTI PRINT
    print(f"Train MSE: {-scores['train_score'].mean():.4f}")
    print(f"Val MSE:   {-scores['test_score'].mean():.4f}")

    row_overfitting_results.append(-scores['train_score'].mean())
    row_overfitting_results.append(-scores['test_score'].mean())

    reg.fit(X_tr, y_tr)

    #Serve poi per explainability
    coefficients_importance.append(reg.coef_)
    
    y_pred = reg.predict(X_te)

    mse, rmse = calculate_errors(y_te, y_pred)
    
    row_mse_results.append(mse)
    row_rmse_results.append(rmse)

    
results_mse.loc[len(results_mse)] = row_mse_results
results_rmse.loc[len(results_rmse)] = row_rmse_results
overfitting_results.loc[len(overfitting_results)] = row_overfitting_results

display(results_mse)
display(results_rmse)
display(overfitting_results)

## Explainability usando Regressione Lineare

Ora valutiamo l'importanza delle features date dalla regressione lineare. Per farlo considero i coefficienti imparati dal modello. I coefficienti in valore assoluto più grandi sono quelli di maggior importanza.

In [ ]:
cs_levels = ['noCS', 'weakCS', 'mildCS', 'strongCS']

for i in range(len(cs_levels)):
    #Voglio creare un dataframe in cui stanno sia i nomi delle features,
    #sia i coefficienti delle stesse
    df_without_target = df_train_noCS.drop(['Z'], axis=1)
    df_importance = pd.DataFrame({'Feature':df_without_target.columns, 'Importance':coefficients_importance[i]})
    df_coef_abs = df_importance['Importance'].abs()
    df_importance['Importance_abs']=df_coef_abs
    df_importance = df_importance.sort_values(by='Importance_abs', ascending=False)
    display(df_importance)



Si nota come variando il grado di CS del dataset, cambi anche l'ordine di importanza delle features valutato tramite i coefficienti di regressione.

La feature più importante è sempre la `zy`, ma le altre variano quasi tutte la propria importanza relativa alla decisione finale del modello.

## Alberi di decisione

In [ ]:
row_mse_results = ["Albero di decisione"]
row_rmse_results = ["Albero di decisione"]
row_overfitting_results = ["Albero di decisione"]

best_estimators = []
for name, group in dataset_groups.items():
    y_tr, X_tr = extract_yX(group["train"])
    y_te, X_te = extract_yX(group['test'])

    X_tr, X_te = standardize_data(X_tr, X_te)#In realtà non serve per alberi di decisione
    
    #Faccio grid search su iperparametri di Albero di decisione
    reg_tree = DecisionTreeRegressor()

    #Scelta dei parametri da provare la faccio considerando che voglio mantenere explainability.
    param_grid = {
        'max_depth': [2, 3, 4, 5],
        'min_samples_leaf': [5, 10, 20],
        'ccp_alpha': [0.0, 0.001, 0.01]
    }
    grid_search = GridSearchCV(reg_tree, param_grid, cv=5, scoring='neg_mean_squared_error')
    
    grid_search.fit(X_tr, y_tr)

    best_estimators.append(grid_search.best_estimator_)
    
    #Dopo aver trovato albero migliore, faccio un'altra cross validatin per verificare presenza di overfitting
    #Infatti grid search non mi tutela da overfitting, se gli iperparametri usati sono errati (es. uso
    #solo profondità di alberi troppo elevate)
    
    kf = KFold(n_splits=5, shuffle=True)
    scores = cross_validate(grid_search.best_estimator_, X_tr, y_tr, cv=kf, scoring="neg_mean_squared_error", return_train_score=True)
    
    print(f"Train MSE: {-scores['train_score'].mean():.4f}")
    print(f"Val MSE:   {-scores['test_score'].mean():.4f}")

    row_overfitting_results.append(-scores['train_score'].mean())
    row_overfitting_results.append(-scores['test_score'].mean())
    
    y_pred = grid_search.best_estimator_.predict(X_te)
    mse, rmse = calculate_errors(y_te, y_pred)
    
    row_mse_results.append(mse)
    row_rmse_results.append(rmse)

    
results_mse.loc[len(results_mse)] = row_mse_results
results_rmse.loc[len(results_rmse)] = row_rmse_results
overfitting_results.loc[len(overfitting_results)] = row_overfitting_results

display(results_mse)
display(results_rmse)
display(overfitting_results)

## Explainability usando Albero di Decisione

Avendo scelto gli iperparametri da testare con grid search in modo tale da ottenere un albero non complesso, e quindi preservare l'interpertabilità della soluzione, ha senso andare a graficare l'albero di decisione risultante:

In [ ]:
def plot_regression_tree(reg_tree, feature_names):
    plt.figure(figsize=(40, 30), dpi=400)  #Aggiusto dimensione e risoluzione
    tree.plot_tree(
        reg_tree,
        filled=True,
        fontsize=10,          # Aumento dimensione font 
        rounded=True,         # Imposto rounded box
        feature_names=feature_names  # Assegno i nomi delle feature
    )
    plt.savefig('testfig.svg', format='svg')
    plt.show()

In [ ]:
feature_names = df_train_noCS.columns.tolist()
for estimator in best_estimators:
    plot_regression_tree(estimator, feature_names)

## Explainable Boosting Machines


In [ ]:
from interpret.glassbox import ExplainableBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV

row_mse_results = ["Explainable Boosting Machine"]
row_rmse_results = ["Explainable Boosting Machine"]
row_overfitting_results = ["Explainable Boosting Machine"]

best_estimators = []
for name, group in dataset_groups.items():
    y_tr, X_tr = extract_yX(group["train"])
    y_te, X_te = extract_yX(group['test'])

    X_tr, X_te = standardize_data(X_tr, X_te)#In realtà non serve per alberi di decisione
    
    ebm = ExplainableBoostingRegressor(n_jobs=-1)

   #Scelta dei parametri da provare la faccio considerando che voglio mantenere explainability
    param_grid = {
        # Più importante - valori limitati per interpretabilità
        "max_leaves": [2, 3],
    
        # Importante per regressione - la doc suggerisce valori alti
        "smoothing_rounds": [350, 500, 750, 1000],
    
        # La doc dice che regressione preferisce learning rate più alto
        "learning_rate": [0.02, 0.03, 0.04, 0.05, 0.1],
    
        # CRITICO per interpretabilità: teniamo basso
        # con 25x o 50x il modello diventa poco leggibile
        "interactions": [0, "1x", "2x", "3x"],
    }

    randomized_search = RandomizedSearchCV(ebm, param_grid, n_iter=20, cv=5, scoring='neg_mean_squared_error', refit=True)
    randomized_search.fit(X_tr, y_tr)

    #Adesso faccio una ulteriore cross validation per verificare che il modello addestrato (il best estimator) non faccia overfitting
    kf = KFold(n_splits=5, shuffle=True)
    scores = cross_validate(randomized_search.best_estimator_, X_tr, y_tr, cv=kf, scoring="neg_mean_squared_error", return_train_score=True)
    
    best_estimators.append(randomized_search.best_estimator_)
    
    print(f"Train MSE: {-scores['train_score'].mean():.4f}")
    print(f"Val MSE:   {-scores['test_score'].mean():.4f}")

    row_overfitting_results.append(-scores['train_score'].mean())
    row_overfitting_results.append(-scores['test_score'].mean())
    
    y_pred = randomized_search.best_estimator_.predict(X_te)
    mse, rmse = calculate_errors(y_te, y_pred)
    
    row_mse_results.append(mse)
    row_rmse_results.append(rmse)

    
results_mse.loc[len(results_mse)] = row_mse_results
results_rmse.loc[len(results_rmse)] = row_rmse_results
overfitting_results.loc[len(overfitting_results)] = row_overfitting_results

display(results_mse)
display(results_rmse)
display(overfitting_results)

## Explainability usando Explainable Boosting Machines

In [ ]:
#Valuto la spiegazione globale del modello. Mi serve a capire l'importanza delle features per interpretabilità.
from interpret import show
for estimator in best_estimators:
    show(estimator.explain_global())

## Verifica di presenza di eteroschedasticità nei dati

### Primo metodo (grafico): residual plot

In [ ]:
for name, df in dfs_train.items():
    print(f"Valutazione di eteroschedasticità nel dataset: {name}")
    
    #Ottengo la lista delle features 
    y_tr, X_tr = extract_yX(df)

    # Fit a linear regression model to get predictions for residuals
    model = LinearRegression()
    model.fit(X_tr, y_tr)
    y_pred_tr = model.predict(X_tr)

    # Calculate residuals
    residuals = y_tr - y_pred_tr
        
    #Creo figure per contenere tutti i plot
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x=y_pred_tr, y=residuals, alpha=0.5)
    plt.axhline(y=0, color='r', linestyle='--')
    plt.title('Residual Plot (Predicted vs. Residuals)')
    plt.xlabel('Predicted Values')
    plt.ylabel('Residuals')
    plt.grid(True, linestyle='--', alpha=0.6)

    filename = f"residual_plots/residuals_{name}.png"
    plt.savefig(filename, bbox_inches='tight')
    
    plt.show()

Tutti i plot visualizzati suggeriscono la presenza di eteroschedasticità nei dati. I residui infatti sono distribuiti non come punti rumorosi randomici, ma hanno invece una forma ben precisa, essendoci patternrn e cluster facilmente individuabili. In altre parole, la dispersione dei residui cambia al variare dei valori predetti, con una forma che in alcune zone è più stretta, ed in altre è più ampia, cosa che indica come la varianza negli errori non sia costante, ma vari.

A questo test grafico affiancherò un test statistico, per avere conferma ulteriore di ciò che emerge. 

### Secondo metodo (statistico): test di Breusch-Pagan

In [ ]:
for name, df in dfs_train.items():
    print(f"Valutazione di eteroschedasticità nel dataset: {name}")
    
    #Ottengo la lista delle features 
    y_tr, X_tr = extract_yX(df)

    X_tr_sm = add_constant(X_tr)
    model_sm = OLS(y_tr, X_tr_sm).fit()

    lm_stat, lm_pvalue, f_stat, f_pvalue = het_breuschpagan(model_sm.resid, model_sm.model.exog)

    print(f"LM statistic: {lm_stat:.5f}")
    print(f"p-value:      {lm_pvalue:.5f}")

Il p-value risultante è inferiore a 0.05 per tutti e 4 i datasets, cosa che indica una forte eteroschedaticità nei dati.

Questo risultato statistico conferma quello grafico ottenuto mediante residual plots, e dunque ci assicura che l'utilizzo del metodo dei Processi Gaussiani per dati eteroschedastici si possa applicare a questo scenario.

Andiamo dunque a implementarlo, per poi valutare le prestazioni confrontandole con quelle degli altri modelli.


## Gaussian Process per dati eteroschedastici

Nel caso di questo modello GP Flow non eseguo una cross validation utilizzando la modalità k-fold garantita dall'implementazione di scikilearn.
Questo perché il modello in questione non implementa l'interfaccia standard di scikitlearn con i metodi .fit e .predict. Una soluzione potrebbe essere quella di creare una class wrapper per il modello che renda possibile l'intergrazione con scikitlearn, ma questo approccio non è molto utilizzato per via di possibili errori come memory_leak di tensorflow che possono accadare in queste circostanze. Un approccio utilizzato 

In [ ]:
row_mse_results = ["GP for Heteroskedastic data"]
row_rmse_results = ["GP for Heteroskedastic data"]
row_overfitting_results = ["GP for Heteroskedastic data"]

for name, group in dataset_groups.items():
    y_tr, X_tr = extract_yX(group["train"])
    y_te, X_te = extract_yX(group['test'])

    X_tr = X_tr.to_numpy().astype(np.float64)
    y_tr = y_tr.to_numpy().astype(np.float64).reshape(-1, 1)
    X_te = X_te.to_numpy().astype(np.float64)
    y_te = y_te.to_numpy().astype(np.float64).reshape(-1, 1)

    X_tr, X_te = standardize_data(X_tr, X_te)

    likelihood = gpf.likelihoods.HeteroskedasticTFPConditional(
    distribution_class=tfp.distributions.Normal,  # Gaussian Likelihood
    scale_transform=tfp.bijectors.Exp(),  # Exponential Transform
    )

    print(f"Likelihood's expected latent_dim: {likelihood.latent_dim}")
    
    
    kernel = gpf.kernels.SeparateIndependent(
        [
            gpf.kernels.SquaredExponential(),  # This is k1, the kernel of f1
            gpf.kernels.SquaredExponential(),  # this is k2, the kernel of f2
        ]
    )
    # The number of kernels contained in gpf.kernels.SeparateIndependent must be the same as likelihood.latent_dim
    
    
    M = 20  # Number of inducing variables for each f_i
    
    # scegli M punti casuali dal training set (considerando che dataset ha più features)
    indices = np.random.choice(len(X_tr), M, replace=False)
    Z = X_tr[indices]
    
    inducing_variable = gpf.inducing_variables.SeparateIndependentInducingVariables(
        [
            gpf.inducing_variables.InducingPoints(Z),  # This is U1 = f1(Z1)
            gpf.inducing_variables.InducingPoints(Z),  # This is U2 = f2(Z2)
        ]
    )
    
    
    model = gpf.models.SVGP(
        kernel=kernel,
        likelihood=likelihood,
        inducing_variable=inducing_variable,
        num_latent_gps=likelihood.latent_dim,
    )
    
    model
    
    
    data = (X_tr, y_tr)
    loss_fn = model.training_loss_closure(data)
    
    gpf.utilities.set_trainable(model.q_mu, False)
    gpf.utilities.set_trainable(model.q_sqrt, False)
    
    variational_vars = [(model.q_mu, model.q_sqrt)]
    natgrad_opt = gpf.optimizers.NaturalGradient(gamma=0.1)
    
    adam_vars = model.trainable_variables
    adam_opt = tf.optimizers.Adam(0.01)
    
    
    @tf.function
    def optimisation_step():
        natgrad_opt.minimize(loss_fn, variational_vars)
    
        with tf.GradientTape() as tape:
            loss = loss_fn()
    
        grads = tape.gradient(loss, adam_vars)
        adam_opt.apply_gradients(zip(grads, adam_vars))
        
        
    epochs = 200
    log_freq = 20
    
    losses = []
    
    for epoch in tqdm(range(1, epochs + 1)):
        optimisation_step()
        losses.append(loss_fn().numpy())
    
        # For every 'log_freq' epochs, print the epoch and plot the predictions against the data
        if epoch % log_freq == 0 and epoch > 0:
            print(f"Epoch {epoch} - Loss: {loss_fn().numpy() : .4f}")
            Ymean, Yvar = model.predict_y(X_tr)
            Ymean = Ymean.numpy().squeeze()
            Ystd = tf.sqrt(Yvar).numpy().squeeze()
    
    model

    from sklearn.metrics import mean_absolute_error

    Ymean_te, Yvar_te = model.predict_y(X_te)
    Ymean_tr, Yvar_tr = model.predict_y(X_tr)

    mse, rmse = calculate_errors(y_te, Ymean_te)
    
    mse = mean_squared_error(y_te, Ymean_te)
    rmse = root_mean_squared_error(y_te, Ymean_te)
    
    print(f"Test MSE: {mse:.4f}")
    print(f"Test RMSE: {rmse:.4f}")

    row_mse_results.append(mse)
    row_rmse_results.append(rmse)

    #PRIMO METODO: GUARDO DIFFERENZA TRA MSE E RSME SU TRAINING E SU VALIDATION TEST
    #In questo metodo GP Flow è spesso più funzionale in pratica valutare l'overfitting non facendo un k-fold-cross-validation, che computazionalmente
    #sarebbe costoso in quanto prevede di riaddestrare il modello gaussiano più volte, ma facendo un singolo split fisso.
    #E' un metodo meno robusto ma che nella pratica è spesso preferibile.
    
    mse = mean_squared_error(y_tr, Ymean_tr)
    rmse = root_mean_squared_error(y_tr, Ymean_tr)
    
    print(f"Train MSE: {mse:.4f}")
    print(f"Train RMSE: {rmse:.4f}")
    
    #SECONDO METODO: GUARDO DIFFERENZA TRA LOGLIKELIHOOD SU TRAINING SET E SU TEST SET
    # TRAIN
    dist_tr = tfp.distributions.Normal(
        loc=Ymean_tr,
        scale=tf.sqrt(Yvar_tr)
    )
    loglik_tr = np.mean(dist_tr.log_prob(y_tr))
    
    # TEST
    dist_te = tfp.distributions.Normal(
        loc=Ymean_te,
        scale=tf.sqrt(Yvar_te)
    )
    loglik_te = np.mean(dist_te.log_prob(y_te))
    
    print(f"Train LogLik: {loglik_tr:.4f}")
    print(f"Test LogLik:  {loglik_te:.4f}")

    #TERZO METODO. TRACCIO LA LOSS DURANTE IL TRAINING:
    plt.plot(losses)
    plt.title("Training Loss")
    plt.show()
    
results_mse.loc[len(results_mse)] = row_mse_results
results_rmse.loc[len(results_rmse)] = row_rmse_results

display(results_mse)
display(results_rmse)

In [ ]:
print(results_mse.columns)
print(row_mse_results)
print(len(results_mse.columns), len(row_mse_results))